In [46]:
%pip install nltk
%pip install websockets
%pip install playwright
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import os
import json
import asyncio
import websockets
import nltk
from datetime import datetime
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from playwright.async_api import async_playwright
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline

In [48]:
# Resource setup
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Download VADER lexicon
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

nltk.download(['punkt', 'stopwords', 'punkt_tab', 'wordnet', 'omw-1.4'])
lemmatizer = WordNetLemmatizer()


sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 # Set to 0 if you have a GPU
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [49]:
BRAVE_PATH = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
AUTH_FILE = "auth.json"
MAX_CONCURRENT_TABS = 3
semaphore = asyncio.Semaphore(MAX_CONCURRENT_TABS)
word_counts = Counter()

# Refined ignore list
ignore_words = [
    "i", "me", "my", "myself", "we", "us", "our", "ours", "ourselves", "you", "your", "yours", 
    "yourself", "yourselves", "he", "him", "his", "she", "her", "hers", "it", "its", "they", 
    "them", "their", "someone", "anyone", "am", "is", "are", "was", "were", "be", "been", 
    "being", "have", "has", "had", "do", "does", "did", "shall", "will", "should", "would", 
    "may", "might", "must", "can", "could", "get", "got", "getting", "go", "goes", "went", 
    "gone", "take", "took", "taken", "make", "made", "making", "like", "know", "knows", 
    "knew", "want", "wants", "wanted", "look", "looks", "really", "very", "just", "actually", 
    "basically", "literally", "even", "also", "still", "maybe", "probably", "quite", "much", 
    "many", "lot", "lots", "thing", "things", "stuff", "reddit", "subreddit", "sub", "post", 
    "posts", "comment", "comments", "thread", "edit", "deleted", "removed", "amp", "https", 
    "http", "com", "www"
]

In [50]:
def get_sentiment(text):
    """
    Scans entire text and understands context using a 
    Transformer model. Returns Positive, Neutral, or Negative.
    """
    # Truncate text to 512 tokens (Model limit)
    truncated_text = text[:512]
    result = sentiment_pipeline(truncated_text)[0]
    
    # Mapping the model labels to your format
    label_map = {
        "positive": "Positive",
        "neutral": "Neutral",
        "negative": "Negative"
    }
    return label_map.get(result['label'].lower(), "Neutral")

In [51]:
def get_cache_path(subreddit):
    # Keep cache local to the Python project
    if not os.path.exists("cache"): os.makedirs("cache")
    return os.path.join("cache", f"{subreddit}.json")

def load_cache(subreddit):
    path = get_cache_path(subreddit)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f: return json.load(f)
    return {}

def save_cache(subreddit, data):
    with open(get_cache_path(subreddit), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [52]:
async def get_post_metadata(post):
    pid = await post.get_attribute("id")
    time_loc = post.locator("time").first
    ts = ""
    if await time_loc.count() > 0:
        ts = await time_loc.get_attribute("datetime")
    return pid, ts

async def scrape_single_post(context, url, pid, ts, websocket, total_priority, tracker, cache, subreddit, is_priority):
    async with semaphore:
        page = await context.new_page()
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            
            title = "[Title Not Found]"
            for sel in ['h1[slot="title"]', 'h1[id^="post-title-"]', 'shreddit-title', 'h1']:
                loc = page.locator(sel).first
                if await loc.is_visible(timeout=3000):
                    title = await loc.inner_text(); break

            content = ""
            for sel in ['shreddit-post-text-body', 'div[slot="text-body"]']:
                loc = page.locator(sel).first
                if await loc.count() > 0:
                    content = await loc.inner_text(timeout=3000)
                    if content: break
            
            # NLP & Sentiment
            full_text = title + " " + content
            sentiment = get_sentiment(full_text)
            words = word_tokenize(full_text.lower())
            sw = set(stopwords.words('english'))
            clean = [lemmatizer.lemmatize(w) for w in words if w.isalnum() and w not in sw and w not in ignore_words]
            keywords = dict(Counter(clean))
            
            # --- PERSIST TO CACHE IMMEDIATELY ---
            post_entry = {
                "id": pid, 
                "timestamp": ts, 
                "title": title, 
                "body": content, 
                "keywords": keywords, 
                "sentiment": sentiment
            }
            cache[pid] = post_entry
            save_cache(subreddit, cache)

            # --- SEND DELTAS ONLY FOR PRIORITY NEW POSTS ---
            if is_priority:
                tracker['ui_processed'] += 1
                progress = int((tracker['ui_processed'] / total_priority) * 100) if total_priority > 0 else 100
                
                if websocket.state.name == 'OPEN':
                    # Send the full post object directly
                    await websocket.send(json.dumps({
                        "type": "delta_update",
                        "post": post_entry,
                        "progress": progress
                    }))
                    print(f"New Post Scraped: {title[:30]} | Progress: {progress}%")
            else:
                print(f"Background Cached (Gap Fill): {title[:30]}")
            
            return post_entry
        except Exception as e:
            print(f"Error {url}: {e}"); return None
        finally: await page.close()

In [53]:
async def run_scraper(websocket, subreddit, count, use_only_cache=False):
    cache = load_cache(subreddit)
    
    # --- SHORT CIRCUIT: CACHE ONLY MODE ---
    if use_only_cache:
        print(f"Cache-only mode requested. Bypassing Playwright for r/{subreddit}.")
        if websocket.state.name == 'OPEN':
            await websocket.send(json.dumps({
                "type": "status", 
                "message": f"Loading cached data for r/{subreddit}..."
            }))
            
            # Sort and send the requested amount from the local file
            final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
            
            await websocket.send(json.dumps({"type": "progress", "value": 100}))
            await websocket.send(json.dumps({
                "type": "final_data", 
                "posts": final_selection
            }))
            await websocket.send(json.dumps({
                "type": "status", 
                "message": "Loaded data exclusively from local cache."
            }))
        return # Terminate the function here. No browsers will open.

    # --- NORMAL MODE: LIVE SCRAPING & BACKGROUND MAINTENANCE ---
    if websocket.state.name == 'OPEN':
        await websocket.send(json.dumps({
            "type": "status", 
            "message": f"Fetching latest {count} posts from r/{subreddit}..."
        }))

    # Determine the safe checkpoint (last_cached marker)
    safe_last_id = None
    for pid, post_data in cache.items():
        if post_data.get("last_cached") is True:
            safe_last_id = pid
            break

    # Fallback for the very first run: start from the newest timestamp if no marker exists
    sorted_cache_list = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)
    if not safe_last_id and sorted_cache_list:
        safe_last_id = sorted_cache_list[0]['id']
    
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False, executable_path=BRAVE_PATH)
    ctx_args = {"storage_state": AUTH_FILE} if os.path.exists(AUTH_FILE) else {}
    context = await browser.new_context(**ctx_args)
    
    try:
        await websocket.send(json.dumps({
            "type": "status", 
            "message": f"Opening r/{subreddit} in the browser..."
        }))
        main_page = await context.new_page()
        await main_page.goto(f"https://www.reddit.com/r/{subreddit}/new", wait_until="domcontentloaded")
        
        # --- PHASE 1: TRUE STREAMING FOR UI (PRIORITY) ---
        priority_tasks = []
        launched_ids = set()
        tracker = {'ui_processed': 0}
        found_safe_last = False
        top_post_id = None

        while len(priority_tasks) < count:
            posts = await main_page.locator('shreddit-post').all()
            new_elements_found = False

            for p in posts:
                if len(priority_tasks) >= count:
                    break

                pid, ts = await get_post_metadata(p)
                if not pid or pid in launched_ids: continue
                
                if not top_post_id:
                    top_post_id = pid 

                launched_ids.add(pid)
                new_elements_found = True

                if pid == safe_last_id:
                    found_safe_last = True

                if pid not in cache:
                    ptype = await p.get_attribute('post-type')
                    if ptype in ['text', 'image', 'gallery', 'link']:
                        href = await p.locator('a[slot="full-post-link"]').get_attribute('href')
                        url = f"https://www.reddit.com{href}"
                        
                        task = asyncio.create_task(
                            scrape_single_post(context, url, pid, ts, websocket, count, tracker, cache, subreddit, True)
                        )
                        priority_tasks.append(task)

            if len(priority_tasks) >= count or found_safe_last:
                break

            if not new_elements_found:
                await main_page.mouse.wheel(0, 4000)
                await asyncio.sleep(1.5)

        if priority_tasks:
            await asyncio.gather(*priority_tasks)

        # --- PHASE 2: SEND FINAL DATA TO UI ---
        if websocket.state.name == 'OPEN':
            final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
            await websocket.send(json.dumps({"type": "progress", "value": 100}))
            await websocket.send(json.dumps({
                "type": "final_data", 
                "posts": final_selection
            }))
            await websocket.send(json.dumps({"type": "status", "message": "Dashboard updated. Background caching started."}))

        # --- PHASE 3: BACKGROUND GAP FILLING (SILENT) ---
        if safe_last_id and not found_safe_last:
            print("Checking for historical gap up to the 'last_cached' marker...")
            background_tasks = []
            found_gap_end = False

            while not found_gap_end and len(background_tasks) < 300: 
                posts = await main_page.locator('shreddit-post').all()

                for p in posts:
                    pid, ts = await get_post_metadata(p)
                    if not pid or pid in launched_ids: continue

                    launched_ids.add(pid)

                    if pid == safe_last_id:
                        found_gap_end = True
                        break

                    if pid not in cache:
                        ptype = await p.get_attribute('post-type')
                        if ptype in ['text', 'image', 'gallery', 'link']:
                            href = await p.locator('a[slot="full-post-link"]').get_attribute('href')
                            url = f"https://www.reddit.com{href}"
                            
                            task = asyncio.create_task(
                                scrape_single_post(context, url, pid, ts, websocket, 0, tracker, cache, subreddit, False)
                            )
                            background_tasks.append(task)

                if found_gap_end: break
                await main_page.mouse.wheel(0, 4000)
                await asyncio.sleep(1.5)

            if background_tasks:
                print(f"Caching {len(background_tasks)} historical posts silently in background...")
                await asyncio.gather(*background_tasks)

        # --- PHASE 4: UPDATE CHECKPOINT MARKER ---
        if top_post_id and top_post_id in cache:
            print(f"Caching successful. Moving 'last_cached' marker to {top_post_id}.")
            
            for post in cache.values():
                post.pop("last_cached", None)
            
            cache[top_post_id]["last_cached"] = True
            save_cache(subreddit, cache)

    except Exception as e:
        print(f"Playwright Execution Error: {e}")
        print("Process interrupted. 'last_cached' marker remains unchanged to preserve gap state.")
        if websocket.state.name == 'OPEN':
            await websocket.send(json.dumps({
                "type": "status", 
                "message": f"Process got interrupted! - {e}"
            }))
    finally:
        await browser.close()
        await playwright.stop()
        print("Session completed.")

In [ ]:
async def handler(websocket):
    print("Waiting for command...")
    async for message in websocket:
        data = json.loads(message)
        if data.get('type') == 'start_scrape':
            sub = data.get('subreddit', 'mumbai')
            limit = data.get('count', 10)
            use_only_cache = data.get('useOnlyCache', False) # Extract the new flag
            
            print(f"Command received: r/{sub} (Limit: {limit}, Cache Only: {use_only_cache})")
            word_counts.clear()
            await run_scraper(websocket, sub, limit, use_only_cache)

async def start_server():
    print("WebSocket Server started on ws://192.168.0.246:8765")
    async with websockets.serve(handler, "192.168.0.246", 8765):
        await asyncio.Future()

await start_server()

WebSocket Server started on ws://192.168.0.246:8765
Waiting for command...
Waiting for command...
Command received: r/Munich (Limit: 100, Cache Only: False)
New Post Scraped: Guys, should I be worried? It  | Progress: 1%
New Post Scraped: Google Maps zeigt jetzt gelösc | Progress: 2%
New Post Scraped: Looking for a place in Munich  | Progress: 3%
New Post Scraped: Gute Laufstrecken in München | Progress: 4%
New Post Scraped: Deutscher Computerspielpreis i | Progress: 5%
New Post Scraped: Rant: People are the worst in  | Progress: 6%
New Post Scraped: Suche zwei Kätzchen zur Adopti | Progress: 7%
New Post Scraped: Kennt wer Gute Nerd oder Furry | Progress: 8%
New Post Scraped: Der Mond ist aufgegangen. | Progress: 9%
New Post Scraped: Public Viewing Schalke vs. Düs | Progress: 10%
New Post Scraped: Looking for a place to train N | Progress: 11%
New Post Scraped: Finally got accepted to Münche | Progress: 12%
New Post Scraped: SBahn access no steps. | Progress: 13%
New Post Scraped: Neue

connection handler failed
Traceback (most recent call last):
  File "c:\python314\Lib\asyncio\windows_events.py", line 463, in finish_socket_func
    return ov.getresult()
           ~~~~~~~~~~~~^^
OSError: [WinError 64] The specified network name is no longer available

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\python314\Lib\asyncio\proactor_events.py", line 286, in _loop_reading
    length = fut.result()
  File "c:\python314\Lib\asyncio\windows_events.py", line 804, in _poll
    value = callback(transferred, key, ov)
  File "c:\python314\Lib\asyncio\windows_events.py", line 467, in finish_socket_func
    raise ConnectionResetError(*exc.args)
ConnectionResetError: [WinError 64] The specified network name is no longer available

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websocke

Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...


opening handshake failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 356, in conn_handler
    await connection.handshake(
    ...<3 lines>...
    )
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 142, in handshake
    async with self.send_context(expected_state=CONNECTING):
               ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python314\Lib\contextlib.py", line 214, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 965, in send_context
    raise self.protocol.close_exc from original_exc
websockets.exceptions.ConnectionClosedError: no close frame received or sent


Waiting for command...
Command received: r/BoycottIsrael (Limit: 100, Cache Only: True)
Cache-only mode requested. Bypassing Playwright for r/BoycottIsrael.


opening handshake failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 356, in conn_handler
    await connection.handshake(
    ...<3 lines>...
    )
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 142, in handshake
    async with self.send_context(expected_state=CONNECTING):
               ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python314\Lib\contextlib.py", line 214, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 965, in send_context
    raise self.protocol.close_exc from original_exc
websockets.exceptions.ConnectionClosedError: no close frame received or sent


Waiting for command...
Waiting for command...
Waiting for command...
Command received: r/AskIndianWomen (Limit: 100, Cache Only: True)
Cache-only mode requested. Bypassing Playwright for r/AskIndianWomen.
Command received: r/AskIndianWomen (Limit: 100, Cache Only: True)
Cache-only mode requested. Bypassing Playwright for r/AskIndianWomen.
Waiting for command...
Waiting for command...
Command received: r/AskIndianWomen (Limit: 100, Cache Only: True)
Cache-only mode requested. Bypassing Playwright for r/AskIndianWomen.
Command received: r/AskIndianWomen (Limit: 100, Cache Only: True)
Cache-only mode requested. Bypassing Playwright for r/AskIndianWomen.
Waiting for command...
Waiting for command...
Command received: r/BoycottIsrael (Limit: 100, Cache Only: True)
Cache-only mode requested. Bypassing Playwright for r/BoycottIsrael.
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Waiting for command...
Command received: r/BoycottIsrael (Limit: 100,